# PRUEBAS

In [0]:
"""
%sql
SELECT COUNT(*) AS total_rows
FROM sbsrisk_dev.silver.silver_sbs_indice_2024;

DESCRIBE EXTENDED sbsrisk_dev.silver.silver_sbs_indice_2024;
"""

In [0]:
"""
from pyspark.sql import functions as F

df24 = spark.table("sbsrisk_dev.bronze.bronze_sbs_morosidad_2024").withColumn("anio", F.lit(2024))
df25 = spark.table("sbsrisk_dev.bronze.bronze_sbs_morosidad_2025").withColumn("anio", F.lit(2025))

df = df24.unionByName(df25, allowMissingColumns=True)

print("Total filas (bronze 2024+2025):", df.count())
display(df.limit(20))
"""

In [0]:
"""
from pyspark.sql import functions as F

df24 = spark.table("sbsrisk_dev.bronze.bronze_sbs_morosidad_2024").withColumn("anio", F.lit(2024))
df25 = spark.table("sbsrisk_dev.bronze.bronze_sbs_morosidad_2025").withColumn("anio", F.lit(2025))
df = df24.unionByName(df25, allowMissingColumns=True)

cols_excel = ["0","1","2","3","4","5","6"]

# conteo de celdas NO vacías por fila
df_chk = df.withColumn(
    "nn",
    sum([F.when(F.col(c).isNotNull() & (F.trim(F.col(c)) != ""), 1).otherwise(0) for c in cols_excel])
)

df_chk.groupBy("nn").count().orderBy("nn").show()
display(df_chk.orderBy(F.desc("nn")).select(*cols_excel, "source_file","periodo_mes","anio","nn").limit(50))
"""

In [0]:
"""
cols_excel = ["0","1","2","3","4","5","6"]

# 1) Quitar filas donde TODAS las columnas estén vacías
cond_all_null = None
for c in cols_excel:
    expr = F.col(c).isNull() | (F.trim(F.col(c)) == "")
    cond_all_null = expr if cond_all_null is None else (cond_all_null & expr)

df_clean = df.filter(~cond_all_null)

# 2) Quitar SOLO filas de "título", pero sin botar las filas con 0 NULL
titulo = F.lower(F.coalesce(F.col("0"), F.lit("")))
df_clean = df_clean.filter(~titulo.contains("boletin informativo mensual"))

# 3) Trim
for c in cols_excel + ["source_file", "periodo_mes", "anio"]:
    df_clean = df_clean.withColumn(c, F.trim(F.col(c)))

print("Total filas silver limpio:", df_clean.count())
display(df_clean.limit(30))
"""

In [0]:
"""(
  df_clean.write.format("delta")
  .mode("overwrite")
  .option("overwriteSchema", "true")
  .saveAsTable("sbsrisk_dev.silver.silver_sbs_indice")
)"""

In [0]:
"""%sql
SELECT anio, COUNT(*) AS filas
FROM sbsrisk_dev.silver.silver_sbs_indice
GROUP BY anio
ORDER BY anio;

SELECT periodo_mes, COUNT(*) AS filas
FROM sbsrisk_dev.silver.silver_sbs_indice
GROUP BY periodo_mes
ORDER BY periodo_mes;"""

In [0]:
display(df_dias_c.limit(10))
display(df_tipo_c.limit(10))

# INICIANDO


In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql import functions as F

catalog = "sbsrisk_dev"
spark.sql(f"USE CATALOG {catalog}")

df2024 = spark.table(f"{catalog}.bronze.bronze_sbs_morosidad_2024")
df2025 = spark.table(f"{catalog}.bronze.bronze_sbs_morosidad_2025")

df_bronze = df2024.unionByName(df2025, allowMissingColumns=True)

print("Total Bronze:", df_bronze.count())
display(df_bronze.select("source_file","periodo_mes","sheet_title","sheet_num").distinct().orderBy("source_file").limit(30))

In [0]:
df_dias = df_bronze.filter(F.lower(F.col("sheet_title")).contains("ratios de morosidad"))
df_tipo = df_bronze.filter(F.lower(F.col("sheet_title")).contains("morosidad por tipo"))

print("dias:", df_dias.count(), "| tipo:", df_tipo.count())

display(df_dias.limit(30))
display(df_tipo.limit(30))

In [0]:
meta_cols = {"source_file","periodo_mes","sheet_title","sheet_num"}

def clean_empty_rows(df):
    data_cols = [c for c in df.columns if c not in meta_cols]
    out = df
    for c in data_cols:
        out = out.withColumn(c, F.when(F.col(c).isin("nan","None",""), None).otherwise(F.col(c)))
    out = out.na.drop("all", subset=data_cols)
    return out

df_dias_c = clean_empty_rows(df_dias)
df_tipo_c = clean_empty_rows(df_tipo)

print("dias limpio:", df_dias_c.count())
print("tipo limpio:", df_tipo_c.count())

display(df_dias_c.limit(30))

In [0]:
# Perfilado - identificación de headers
candidates = (df_dias_c
  .select("source_file","periodo_mes","sheet_title","sheet_num", F.col("0").alias("c0"), F.col("1").alias("c1"), F.col("2").alias("c2"))
  .withColumn("c0l", F.lower(F.col("c0")))
  .withColumn("c1l", F.lower(F.col("c1")))
  .filter(
      F.col("c0l").contains("ratio") |
      F.col("c0l").contains("días") |
      F.col("c1l").contains("días") |
      F.col("c0l").contains("caja") |
      F.col("c1l").contains("caja")
  )
)

display(candidates.limit(50))

In [0]:
spark.sql(f"USE SCHEMA silver")

silver_dias_path = f"abfss://silver@adlsbsriskdev.dfs.core.windows.net/{catalog}/silver/silver_morosidad_dias_stg"
silver_tipo_path = f"abfss://silver@adlsbsriskdev.dfs.core.windows.net/{catalog}/silver/silver_morosidad_tipo_stg"

(df_dias_c.write.format("delta").mode("overwrite").option("overwriteSchema","true").save(silver_dias_path))
(df_tipo_c.write.format("delta").mode("overwrite").option("overwriteSchema","true").save(silver_tipo_path))

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.silver.silver_morosidad_dias_stg
USING DELTA
LOCATION '{silver_dias_path}'
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.silver.silver_morosidad_tipo_stg
USING DELTA
LOCATION '{silver_tipo_path}'
""")

spark.sql(f"SELECT COUNT(*) total FROM {catalog}.silver.silver_morosidad_dias_stg").show()
spark.sql(f"SELECT COUNT(*) total FROM {catalog}.silver.silver_morosidad_tipo_stg").show()